# Client segments, flow correlation, concentration and liquidity stress

This notebook works through the eight-phase brief: inventory, data checks, fund context, client features, segmentation, segment flow correlation, concentration, stress tests and write-up. All logic lives in `notebooks/segment_stress.py`. **Every table and column name is taken from one `CONFIG` dictionary (Phase 0)**, so the notebook runs on a different fund database by editing a single cell.

Amounts are in GBP. Behaviour, segments, correlation and stress use **trade dates** (when the instruction is given). Cash-date reconciliation is checked separately. The data are synthetic.

In [1]:
from pathlib import Path
import sys, json
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg', force=True)
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
ROOT = Path.cwd() if (Path.cwd()/'synthetic_fund_daily.csv').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT/'notebooks'))
import segment_stress as S
FIG = ROOT/'outputs/figures'
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
gbp = lambda v: f'£{v/1e6:,.1f}m'
pct = lambda v: f'{100*v:.1f}%'

## Phase 0 · Inventory and mapping

The analysis needs four kinds of information. The configuration below maps this database onto them: column names, date format, the sign convention for redemptions, which statuses count, and which date drives behaviour. **To run on another database, edit only this cell.**

In [2]:
CONFIG = S.CONFIG   # edit S.CONFIG fields here for another database, e.g. CONFIG = {**S.CONFIG, 'files': {...}}
print(json.dumps({k: v for k, v in CONFIG.items() if k in ['name', 'files', 'flows', 'redemption_sign', 'keep_status', 'behaviour_date', 'currency']}, indent=1, default=str))
o = S.run(CONFIG, write=True)
display(o['inventory'])

{
 "name": "Charity fund (synthetic, 2006\u20132026)",
 "files": {
  "clients": "synthetic_clients.csv",
  "flows": "synthetic_client_flows.csv",
  "fund": "synthetic_fund_daily.csv",
  "payables": "synthetic_income_distributions.csv"
 },
 "flows": {
  "id": "transaction_id",
  "client": "client_id",
  "trade_date": "trade_date",
  "cash_date": "cash_flow_date",
  "amount": "flow_gbp",
  "status": "cash_flow_status",
  "balance_after": "client_aum_after_trade_gbp",
  "units_after": null,
  "currency": null,
  "flow_type": null
 },
 "redemption_sign": -1,
 "keep_status": [
  "Settled"
 ],
 "behaviour_date": "trade_date",
 "currency": "GBP"
}


,role,table,rows,fields_present,fields_missing,notes
0,Client master,synthetic_clients.csv,11103,"id, sector, open, close, share_class, balance",—,
1,Transactions,synthetic_client_flows.csv,901088,"client, trade_date, amount, status, balance_after",—,
2,Fund daily,synthetic_fund_daily.csv,5053,"date, aum",—,"unit prices: Income, Accumulation"
3,Known future outflows,synthetic_income_distributions.csv,462696,"client, date, amount",—,


All four roles are present. Unit prices exist for both share classes, so holdings can be rebuilt as units × price. There is no units column, so units are derived from balance-after-trade ÷ class price. There is no flow-type or currency column, so internal moves and currencies are checked heuristically.

## Phase 1 · Data checks

The analysis does not continue until client flows reconcile to fund flows and holdings reconcile to units × price.

In [3]:
display(o['checks'])
assert o['checks'].passed.all(), 'Data checks failed: stop here'
print('Unit reconstruction:', {k: round(v, 6) for k, v in o['unit_check'].items()})

,check,result,detail,passed
0,Status filter,"kept 901,088 of 901,088","{""Settled"": 901088}",True
1,Duplicates,"0 duplicate IDs, 0 duplicate rows",exact duplicates removed,True
2,Internal moves,8884 same-day equal-and-opposite amount groups,"No flow-type column; candidates reported, not ...",True
3,Currencies,GBP,single currency,True
4,Omnibus / nominee accounts,0 by type name; 0 top-1% holders with extreme ...,none hidden behind single IDs detected,True
5,Trade vs cash date,"behaviour, segments, correlation and stress us...",cash-date totals reconciled separately; paymen...,True
6,Client flows = fund flows (trade date),max daily difference £0.00,22 transactions dated outside fund calendar (p...,True
7,Client flows = fund flows (cash date),max daily difference £0.00,0 transactions dated outside fund calendar (pr...,True
8,Holdings = units × price,median unit error 0.00023; rebuilt fund AUM wi...,units from balance-after-trade ÷ class price; ...,True


Unit reconstruction: {'median_abs_unit_error': 0.000233, 'p99_abs_unit_error': 0.001574, 'max_relative_error': 0.002569}


* **Status:** every transaction is settled. The filter is kept so that cancelled, pending or reversed rows would be dropped in another database.
* **Duplicates:** none by ID or by content.
* **Internal moves:** the database has no flow-type column. Same-day equal-and-opposite amounts between different clients are counted as *candidates* but not removed. With 900k transactions, many small amounts coincide by chance. In a database with switch or transfer codes, set `flows.flow_type` and these rows are netted out.
* **Omnibus / nominee accounts:** no client type suggests a platform or nominee, and no top-1% holder has an extreme trade count.
* **Currency:** single currency (GBP).
* **Trade vs cash date:** client flows reconcile to fund flows on *both* dates to the penny. Behaviour uses trade date; cash timing (settlement lag) matters only for when cash leaves.
* **Holdings:** daily units × price reproduce reported fund AUM every day, within rounding. Segment AUM history (Phase 5) depends on this.

## Phase 2 · Fund context and stress windows

The fund type decides how stress is defined. A money market fund's price is flat, so it cannot signal stress, and only external markers (named episodes, market indicators) would be used. This fund's price is volatile, so it is **not** an MMF. Stress days are those where the accumulation price is at least 10% below its trailing one-year high, **plus** named market episodes (GFC, euro-area crisis, COVID-19, the 2022 UK gilt/LDI crisis).

In [4]:
display(pd.Series(o['ctx']))
m = o['month']; fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].bar(m.month - .2, m.sub_rate_bp, .4, label='Subscriptions'); ax[0].bar(m.month + .2, m.red_rate_bp, .4, label='Redemptions')
ax[0].set_xticks(range(1, 13)); ax[0].set_ylabel('Daily flow, bp of AUM (summed over history)'); ax[0].set_title('Seasonality by calendar month'); ax[0].legend()
fl = o['flags']; ax[1].plot(fl.index, 100*fl.drawdown_1y, lw=.8, color='#0f2a4a'); ax[1].fill_between(fl.index, -35, 0, where=fl.stress, color='#e34948', alpha=.2, label='Stress day')
ax[1].set_ylabel('Price vs 1-year high (%)'); ax[1].set_title('Stress windows'); ax[1].legend(); fig.tight_layout(); fig.savefig(FIG/'11_context.png', dpi=110); plt.show()
display(m.round(2))

annualised_price_vol                                              0.10327
max_drawdown                                                    -0.304362
fund_type                                        Non-MMF (price-variable)
stress_definition       fund accumulation price ≥10% below its trailin...
stress_days                                                           781
calm_days                                                            4272
dtype: object

/var/folders/bc/9np4wbzx4nx361l0f218skn00000gn/T/ipykernel_35906/3336738090.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax[1].set_ylabel('Price vs 1-year high (%)'); ax[1].set_title('Stress windows'); ax[1].legend(); fig.tight_layout(); fig.savefig(FIG/'11_context.png', dpi=110); plt.show()


,month,sub,red,aum,sub_rate_bp,red_rate_bp,net_rate_bp
0,1,8.143719e+08,8.147908e+07,7.112234e+11,11.45,1.15,10.30
1,2,5.401391e+08,1.489383e+08,6.885119e+11,7.85,2.16,5.68
2,3,1.251095e+08,1.065893e+09,7.347417e+11,1.70,14.51,-12.80
3,4,5.323621e+08,1.611616e+08,6.580279e+11,8.09,2.45,5.64
4,5,5.475751e+08,9.133169e+07,6.857377e+11,7.99,1.33,6.65
5,6,3.753528e+08,6.375182e+08,7.288072e+11,5.15,8.75,-3.60
6,7,3.258681e+08,4.742351e+08,7.592693e+11,4.29,6.25,-1.95
7,8,1.513751e+08,8.746852e+07,7.242869e+11,2.09,1.21,0.88
8,9,6.064915e+08,6.451612e+08,7.485226e+11,8.10,8.62,-0.52
9,10,6.568781e+08,1.787959e+08,7.825482e+11,8.39,2.28,6.11


## Phase 3 · Client features

The core features are holding size, trades per year, redemption share, trade size relative to holding, largest redemption (relative to the average holding) and **stress lift**. Stress lift is a client's redemption rate on stress days divided by its rate on calm days, shrunk towards 1 so that small clients do not produce extreme ratios.

The calendar features fit a charity client base: the share of redemptions in March and in December (common financial year-ends), and the share of trades in the last three business days of a month or quarter. Corporate clients would instead get tax and payroll dates. **Yield sensitivity** (flows against the gap between fund yield and Bank Rate) applies to MMFs and needs a yield series, which this database lacks, so it is left empty.

**Eligibility rule recheck.** The table shows how many clients pass different thresholds of trades and years, and how much AUM the excluded *rare traders* hold.

In [5]:
display(o['eligibility'].assign(eligible_share=lambda d: d.eligible_share.map(pct), rare_aum_share=lambda d: d.rare_aum_share.map(pct)))
f = o['features']; print(f"Rule used: >= {CONFIG['eligibility']['min_trades']} trades over >= {CONFIG['eligibility']['min_years']} year(s): {f.eligible.sum():,} eligible; rare traders hold {pct(f.loc[~f.eligible,'current_holding_gbp'].sum()/f.current_holding_gbp.sum())} of AUM")

,min_trades,min_years,eligible_clients,eligible_share,rare_aum_share
0,2,0.5,10754,96.9%,2.2%
1,2,1.0,10681,96.2%,2.5%
2,2,2.0,10496,94.5%,3.7%
3,4,0.5,10004,90.1%,10.3%
4,4,1.0,9941,89.5%,10.6%
5,4,2.0,9777,88.1%,11.7%
6,8,0.5,8285,74.6%,27.3%
7,8,1.0,8269,74.5%,27.3%
8,8,2.0,8136,73.3%,28.1%
9,12,0.5,7737,69.7%,33.0%


Rule used: >= 8 trades over >= 1.0 year(s): 8,269 eligible; rare traders hold 27.3% of AUM


Clients are many and active (median 75 trades), so the original **8 trades over at least 1 year** bar does not need lowering for this fund. Rare traders are **kept as their own segment** rather than dropped, because they hold about 27% of AUM, the same share found last time.

## Phase 4 · Segmentation

Features are log-transformed where skewed, standardised, then clustered with k-means for k = 3–8. k is chosen as the best silhouette among *usable* solutions (every segment at least 3% of eligible clients). **Stability:** k-means is refitted on bootstrap resamples and every client is reassigned. The adjusted Rand index (ARI) measures agreement with the reference labels (1 = identical, 0 = chance).

In [6]:
display(o['selection'].round(3))
a = o['ari']; print(f"Chosen k = {int(o['selection'].query('chosen').k.iloc[0])}; bootstrap ARI over {len(a)} resamples: median {np.median(a):.2f}, 10th percentile {np.percentile(a,10):.2f}, min {a.min():.2f}")
p = o['profiles'].copy(); display(p.assign(aum_gbp=p.aum_gbp.map(gbp), aum_share=p.aum_share.map(pct), median_holding=p.median_holding.map(lambda v: f'£{v:,.0f}')).round(3))

,k,silhouette,smallest_segment_share,inertia,ari_median_10boot,chosen
0,3,0.158,0.134,57706.146,0.782,False
1,4,0.142,0.058,52834.727,0.891,False
2,5,0.142,0.056,49318.697,0.566,False
3,6,0.148,0.055,46043.740,0.852,False
4,7,0.161,0.050,43513.323,0.718,True
5,8,0.161,0.046,41366.669,0.726,False


Chosen k = 7; bootstrap ARI over 30 resamples: median 0.72, 10th percentile 0.67, min 0.65


,segment,clients,aum_gbp,median_holding,trades_per_year,redemption_share,trade_rel_holding,largest_red_rel,stress_lift,march_red_share,december_red_share,month_end_share,aum_share
0,Large active holders,2149,"£1,195.0m","£83,964",6.852,0.463,0.046,0.137,1.003,0.254,0.236,0.040,39.8%
1,Low-activity holders,416,£150.4m,"£33,893",0.450,0.433,0.054,0.131,0.998,0.055,0.051,0.153,5.0%
2,March (year-end) redeemers,585,£105.9m,"£28,658",4.523,0.444,0.049,0.140,1.015,0.459,0.020,0.048,3.5%
3,Month-end traders,780,£96.9m,"£21,620",6.470,0.448,0.033,0.153,0.998,0.219,0.110,0.253,3.2%
4,Net withdrawers,918,£44.1m,"£25,456",6.277,0.559,0.045,1.054,0.934,0.098,0.045,0.057,1.5%
5,Rare traders,2834,£819.8m,"£32,286",0.312,0.202,0.056,0.069,0.998,0.098,0.073,0.147,27.3%
6,Steady accumulators,1821,£540.3m,"£44,017",6.102,0.404,0.028,0.104,0.990,0.151,0.026,0.059,18.0%
7,Year-end redeemers,1600,£47.6m,"£12,897",7.052,0.508,0.108,0.603,1.001,0.235,0.213,0.043,1.6%


**Naming.** Segments are named from their profile: the feature on which each segment stands out most, in standard deviations.
**Validation against client attributes** (Cramér's V: 0 = unrelated, 1 = determined):

In [7]:
display(o['validation'].round(3)); display((100*o['sector_mix']).round(0))
fig, ax = plt.subplots(figsize=(14, 4)); v = o['seg_aum'].resample('M').last()/1e9
ax.stackplot(v.index, v.T.values, labels=v.columns); ax.set_ylabel('AUM (£bn)'); ax.set_title('Rebuilt AUM by segment (units × price)'); ax.legend(loc='upper left', fontsize=8, ncol=4)
fig.tight_layout(); fig.savefig(FIG/'11_segment_aum.png', dpi=110); plt.show()

,attribute,cramers_v,categories
0,client_sector,0.355,11
1,client_type,0.358,34
2,unit_type,0.114,2
3,country,0.019,4


client_sector,Amateur sport,Arts and heritage,Community and social welfare,Environment and animal welfare,Faith and religion,Grant-making foundations,Health and hospices,Higher education,Housing and almshouses,International aid,Schools
row_0,,,,,,,,,,,
Large active holders,0.0,0.0,23.0,0.0,2.0,16.0,16.0,4.0,11.0,0.0,29.0
Low-activity holders,4.0,4.0,16.0,7.0,25.0,14.0,11.0,0.0,3.0,4.0,11.0
March (year-end) redeemers,32.0,1.0,4.0,44.0,13.0,1.0,1.0,0.0,0.0,1.0,2.0
Month-end traders,1.0,1.0,22.0,1.0,56.0,3.0,11.0,1.0,1.0,0.0,3.0
Net withdrawers,3.0,18.0,8.0,3.0,36.0,5.0,5.0,1.0,2.0,14.0,6.0
Rare traders,4.0,5.0,13.0,5.0,26.0,13.0,10.0,1.0,5.0,5.0,12.0
Steady accumulators,1.0,17.0,2.0,2.0,59.0,2.0,2.0,0.0,1.0,13.0,0.0
Year-end redeemers,1.0,1.0,19.0,2.0,6.0,32.0,18.0,1.0,7.0,0.0,14.0


/var/folders/bc/9np4wbzx4nx361l0f218skn00000gn/T/ipykernel_35906/22160038.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIG/'11_segment_aum.png', dpi=110); plt.show()


**Why segments rather than sector?** A fair, out-of-time test: segments are formed from behaviour in 2006–2016. Then segments, sector, client type and region are each scored on how much of every client's 2016–2026 behaviour they explain (η², the between-group share of variance), for clients eligible in both periods.

In [8]:
g = o['grouping'].pivot(index='feature', columns='grouping', values='eta_squared'); display((100*g).round(1)); display((100*g.mean()).round(1).rename('mean % explained'))
print(f"{o['grouping_clients']:,} clients eligible in both periods")

grouping,Behavioural segments (formed 2006–2016),Client type,Region,Sector
feature,,,,
log_holding,25.7,9.3,0.1,7.9
log_largest_red_rel,20.5,12.8,0.2,10.5
log_stress_lift,2.3,5.3,0.2,5.0
log_trade_rel,40.0,29.5,0.2,25.5
log_trades_per_year,68.3,15.8,0.2,15.4
month_end_trade_share,54.1,15.1,0.1,14.8
red_december_share,38.8,68.1,0.1,68.0
red_march_share,26.8,57.1,0.1,56.7
redemption_share,2.0,5.8,0.2,2.5


grouping
Behavioural segments (formed 2006–2016)    31.0
Client type                                24.3
Region                                      0.2
Sector                                     22.9
Name: mean % explained, dtype: float64

5,816 clients eligible in both periods


Segments explain more of later behaviour overall, especially trading frequency, trade and holding size and month-end habits (*how* clients use the fund). Sector and client type explain year-end timing (March and December redemptions) better: *when* clients redeem is set by the type of charity. Region explains almost nothing. The practical conclusion is to use both: plan timing by sector, and judge size and lumpiness by segment.

## Phase 5 · Segment flow correlation

Each segment's daily AUM is rebuilt from units × price. Its daily flow rate is net flow divided by the previous day's AUM. Correlation between segments is compared four ways: **raw vs deseasonalised** (month × part-of-month and weekday effects removed), and **calm vs stress**. Because runs on liquid funds develop within days, it is computed on **daily** data as well as weekly. A week counts as a stress week if it has 3 or more stress days.

**Stress response by value.** The client-level stress lift above is a median, so it describes the *typical* client. The table below measures each segment's total redemptions as a share of its AUM on stress days versus calm days, and versus calm days in the **same calendar month** (so a stress period that happens to fall in March is not mistaken for stress behaviour).

In [9]:
display(o['value_lift'].round(3))

,segment,stress_bp_per_day,calm_bp_per_day,value_lift,month_matched_value_lift
0,Large active holders,9.238,7.767,1.189,1.113
1,Low-activity holders,1.059,0.723,1.464,1.423
2,March (year-end) redeemers,3.619,3.275,1.105,0.761
3,Month-end traders,9.121,7.406,1.232,1.196
4,Net withdrawers,7.761,11.932,0.650,0.676
5,Rare traders,0.816,0.601,1.357,1.363
6,Steady accumulators,3.338,3.474,0.961,0.991
7,Year-end redeemers,34.351,27.315,1.258,1.182


In [10]:
display(o['corr_summary'].round(3))
fig, ax = plt.subplots(1, 4, figsize=(18, 4.2))
for a_, key in zip(ax, [('daily','raw','calm'), ('daily','raw','stress'), ('daily','deseasonalised','calm'), ('daily','deseasonalised','stress')]):
    cm = o['corr'][key]; im = a_.imshow(cm, vmin=-1, vmax=1, cmap='RdBu'); a_.set_title(' · '.join(key), fontsize=9)
    a_.set_xticks(range(len(cm)), [c[:12] for c in cm.columns], rotation=90, fontsize=7); a_.set_yticks(range(len(cm)), [c[:14] for c in cm.index], fontsize=7)
fig.colorbar(im, ax=ax, shrink=.8); fig.savefig(FIG/'11_segment_correlation.png', dpi=110, bbox_inches='tight'); plt.show()

,frequency,basis,regime,periods,mean_pairwise_r,min_r,max_r
0,daily,raw,all,5052,0.109,-0.025,0.538
1,daily,raw,calm,4271,0.107,-0.030,0.551
2,daily,raw,stress,781,0.125,-0.041,0.480
3,daily,deseasonalised,all,5052,0.048,-0.022,0.231
4,daily,deseasonalised,calm,4271,0.047,-0.028,0.247
5,daily,deseasonalised,stress,781,0.055,-0.041,0.232
6,weekly,raw,all,1044,0.185,-0.035,0.734
7,weekly,raw,calm,886,0.178,-0.047,0.744
8,weekly,raw,stress,158,0.216,-0.078,0.693
9,weekly,deseasonalised,all,1044,0.092,-0.057,0.419


/var/folders/bc/9np4wbzx4nx361l0f218skn00000gn/T/ipykernel_35906/762258017.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.colorbar(im, ax=ax, shrink=.8); fig.savefig(FIG/'11_segment_correlation.png', dpi=110, bbox_inches='tight'); plt.show()


## Phase 6 · Concentration

Holdings at each year-end (and at the latest date) are rebuilt as units × price. The table shows the share held by the top 1, 5, 10, 20, 50 and 100 clients, and how many of each top-N group are rare traders.

In [11]:
c = o['concentration']; display(c[['date','holders']+[f'top{n}_share' for n in [1,5,10,20,50,100]]].assign(**{f'top{n}_share': c[f'top{n}_share'].map(pct) for n in [1,5,10,20,50,100]}).tail(8))
display(c[['date']+[f'top{n}_rare_traders' for n in [1,5,10,20,50,100]]].tail(3))
t = o['top']; display(t.head(20).assign(holding_gbp=t.holding_gbp.map(gbp), share=t.share.map(pct)))
display(t.groupby('segment').agg(clients_in_top100=('client_id','size'), holding=('holding_gbp','sum')).assign(holding=lambda d: d.holding.map(gbp)))

,date,holders,top1_share,top5_share,top10_share,top20_share,top50_share,top100_share
13,2019-12-31,9066,2.1%,8.3%,14.5%,21.1%,33.4%,42.4%
14,2020-12-31,9235,2.3%,8.6%,14.4%,20.8%,32.6%,41.8%
15,2021-12-31,9376,2.5%,8.6%,14.5%,21.0%,33.1%,42.3%
16,2022-12-30,9514,2.4%,8.4%,14.2%,20.9%,32.8%,42.1%
17,2023-12-29,9642,2.4%,8.4%,14.0%,20.9%,32.8%,41.9%
18,2024-12-31,9788,2.3%,8.2%,13.9%,20.8%,32.8%,41.9%
19,2025-12-31,9909,2.1%,8.1%,13.5%,20.5%,32.3%,41.5%
20,2026-10-30,9995,2.1%,8.0%,13.4%,20.4%,31.8%,40.7%


,date,top1_rare_traders,top5_rare_traders,top10_rare_traders,top20_rare_traders,top50_rare_traders,top100_rare_traders
18,2024-12-31,0,4,6,8,14,24
19,2025-12-31,0,3,6,7,15,25
20,2026-10-30,0,3,6,8,14,24


,client_id,holding_gbp,share,segment,rank
0,C005234,£63.7m,2.1%,Large active holders,1
1,C012938,£47.9m,1.6%,Rare traders,2
2,C002028,£44.9m,1.5%,Rare traders,3
3,C004848,£41.5m,1.4%,Large active holders,4
4,C019283,£41.3m,1.4%,Rare traders,5
5,C024454,£38.8m,1.3%,Rare traders,6
6,C024578,£35.6m,1.2%,Rare traders,7
7,C024418,£29.8m,1.0%,Large active holders,8
8,C027569,£28.8m,1.0%,Rare traders,9
9,C020866,£28.5m,1.0%,Low-activity holders,10


,clients_in_top100,holding
segment,,
Large active holders,48,£564.1m
Low-activity holders,6,£79.0m
March (year-end) redeemers,2,£24.1m
Month-end traders,1,£5.0m
Rare traders,24,£404.3m
Steady accumulators,19,£143.6m


## Phase 7 · Stress tests

Five families of scenario are measured against AUM at 30 October 2026:
1. **Concentration:** the top N clients redeem in full.
2. **Historical replay:** the worst observed net outflow over 1, 5 and 20 days (as a share of AUM) and the worst inside each named episode, applied to today's AUM.
3. **Segment tails:** each segment's 99th-percentile worst outflow rate, either assuming all segments hit together (ρ = 1) or combined with the historical *stress-period* correlation matrix.
4. **Combined:** a 5-day tail plus the top-10 exit plus known payables (declared distributions and redemptions awaiting payment).
5. **Reverse stress test:** how many of the largest clients could leave before a liquidity buffer is exhausted.

**What is tested against.** This database has no portfolio holdings or maturities, so daily and weekly liquid assets cannot be measured. Buffers of 5/10/20% of AUM are **illustrative**. The money-market regulatory floors are shown for reference only, because this fund is not an MMF. UK MMFR currently requires 10% daily / 30% weekly liquid assets for LVNAV/CNAV funds and 7.5% / 15% for short-term VNAV funds. US Rule 2a-7 (2023) requires 25% / 50%. In June 2026 the FCA said it would keep the current UK minimums rather than raise them as proposed in CP23/28; further guidance is pending. **Second round:** once a 10% liquid buffer is used up, the rest must come from asset sales. The loss at illustrative fire-sale haircuts of 1/3/5% is shown.

In [12]:
T = o['tests']; display(T.assign(outflow_gbp=T.outflow_gbp.map(gbp), outflow_share=T.outflow_share.map(pct), payables_included_gbp=T.payables_included_gbp.map(gbp)))
display(o['tails'].assign(tail_rate=lambda d: d.tail_rate.map(pct), segment_aum_gbp=lambda d: d.segment_aum_gbp.map(gbp), tail_outflow_gbp=lambda d: d.tail_outflow_gbp.map(gbp)))
display(o['reverse'].assign(buffer_gbp=lambda d: d.buffer_gbp.map(gbp), after_payables_gbp=lambda d: d.after_payables_gbp.map(gbp)))
sr = o['second_round']; display(sr[sr.shortfall_gbp > 0].assign(shortfall_gbp=lambda d: d.shortfall_gbp.map(gbp), sale_needed_gbp=lambda d: d.sale_needed_gbp.map(gbp), loss_gbp=lambda d: d.loss_gbp.map(gbp)))
print('Base:', o['base'])

,family,scenario,outflow_gbp,outflow_share,payables_included_gbp
0,Concentration,Top 1 clients redeem in full,£63.7m,2.1%,£0.0m
1,Concentration,Top 5 clients redeem in full,£239.4m,8.0%,£0.0m
2,Concentration,Top 10 clients redeem in full,£400.8m,13.4%,£0.0m
3,Concentration,Top 20 clients redeem in full,£611.6m,20.4%,£0.0m
4,Concentration,Top 50 clients redeem in full,£954.0m,31.8%,£0.0m
5,Historical replay,Worst 1-day net outflow (ended 2020-12-01),£35.4m,1.2%,£0.0m
6,Historical replay,Worst 5-day net outflow (ended 2013-03-22),£62.7m,2.1%,£0.0m
7,Historical replay,Worst 20-day net outflow (ended 2012-03-30),£112.5m,3.8%,£0.0m
8,Named episode,Global financial crisis: worst 5-day net outflow,£58.5m,2.0%,£0.0m
9,Named episode,Global financial crisis: worst 20-day net outflow,£100.6m,3.4%,£0.0m


,horizon_days,segment,tail_rate,segment_aum_gbp,tail_outflow_gbp
0,1,Large active holders,0.8%,"£1,195.0m",£10.0m
1,1,Low-activity holders,0.1%,£150.4m,£0.1m
2,1,March (year-end) redeemers,0.4%,£105.9m,£0.4m
3,1,Month-end traders,1.1%,£96.9m,£1.1m
4,1,Net withdrawers,0.8%,£44.1m,£0.4m
5,1,Rare traders,0.1%,£816.3m,£0.8m
6,1,Steady accumulators,0.3%,£540.3m,£1.9m
7,1,Year-end redeemers,2.2%,£47.6m,£1.0m
0,5,Large active holders,3.1%,"£1,195.0m",£37.1m
1,5,Low-activity holders,0.7%,£150.4m,£1.1m


,buffer,buffer_gbp,after_payables_gbp,clients_before_exhausted,note
0,5% of AUM,£150.0m,£138.1m,2,
1,10% of AUM,£300.0m,£288.1m,6,
2,20% of AUM,£600.0m,£588.1m,18,
3,UK MMFR · LVNAV/CNAV (current) daily floor 10.0%,£300.0m,£288.1m,6,reference only: this fund is not an MMF
4,UK MMFR · LVNAV/CNAV (current) weekly floor 30.0%,£900.0m,£888.1m,42,reference only: this fund is not an MMF
5,UK MMFR · VNAV short-term (current) daily floo...,£225.0m,£213.1m,4,reference only: this fund is not an MMF
6,UK MMFR · VNAV short-term (current) weekly flo...,£450.0m,£438.1m,11,reference only: this fund is not an MMF
7,US Rule 2a-7 (2023 amendments) daily floor 25.0%,£750.0m,£738.1m,28,reference only: this fund is not an MMF
8,US Rule 2a-7 (2023 amendments) weekly floor 50.0%,"£1,500.0m","£1,488.1m",100,reference only: this fund is not an MMF


,scenario,family,shortfall_gbp,haircut,sale_needed_gbp,loss_gbp
6,Top 10 clients redeem in full,Concentration,£100.8m,0.01,£101.8m,£1.0m
7,Top 10 clients redeem in full,Concentration,£100.8m,0.03,£103.9m,£3.1m
8,Top 10 clients redeem in full,Concentration,£100.8m,0.05,£106.1m,£5.3m
9,Top 20 clients redeem in full,Concentration,£311.6m,0.01,£314.8m,£3.1m
10,Top 20 clients redeem in full,Concentration,£311.6m,0.03,£321.2m,£9.6m
11,Top 20 clients redeem in full,Concentration,£311.6m,0.05,£328.0m,£16.4m
12,Top 50 clients redeem in full,Concentration,£654.0m,0.01,£660.6m,£6.6m
13,Top 50 clients redeem in full,Concentration,£654.0m,0.03,£674.3m,£20.2m
14,Top 50 clients redeem in full,Concentration,£654.0m,0.05,£688.5m,£34.4m
60,Worst 5-day replay + top 10 full exit + known ...,Combined,£175.4m,0.01,£177.2m,£1.8m


Base: {'aum': 3000000006.93, 'date': '2026-10-30', 'known_payables': 11909421.12, 'payable_columns': {'income_distributions_payable_gbp': 11909421.12, 'redemptions_awaiting_payment_gbp': -0.0}, 'scheduled_distributions_after_date': 11909421.120000001}


## Phase 8 · Findings

Outputs are this executed notebook, its HTML export (`outputs/11_segments_correlation_stress.html`), the client-to-segment file (`outputs/tables/11_client_segments.csv`), all tables (`outputs/tables/11_*.csv`) and the dashboard page *Segments & stress*. The findings below are generated from the results, so they update when the configuration changes.

In [13]:
p = o['profiles'].set_index('segment'); c = o['concentration'].iloc[-1]; T = o['tests'].set_index('scenario'); cs = o['corr_summary'].set_index(['frequency','basis','regime']).mean_pairwise_r
vl = o['value_lift']; rare = p.loc['Rare traders']; biggest = p.drop('Rare traders').aum_share.idxmax(); lift = p.stress_lift
rev = o['reverse'].set_index('buffer').clients_before_exhausted
w5 = T[(T.family=='Historical replay') & T.index.str.contains('5-day')].outflow_gbp.max()
txt = f'''
1. **Data reconcile.** Client flows equal fund flows to the penny on trade and cash dates, and units × price rebuild fund AUM every day. No cancelled, duplicate, multi-currency or omnibus records were found. Internal moves cannot be identified without a flow-type code.
2. **Fund type.** {o['ctx']['fund_type']}: annualised price volatility {pct(o['ctx']['annualised_price_vol'])}, maximum drawdown {pct(o['ctx']['max_drawdown'])}. Stress is defined from the price plus named episodes ({o['ctx']['stress_days']:,} stress days).
3. **Segments.** k = {int(o['selection'].query('chosen').k.iloc[0])} behavioural segments plus rare traders. Silhouette is low ({o['selection'].query('chosen').silhouette.iloc[0]:.2f}): client behaviour is a continuum rather than sharply separated groups. Assignments are fairly stable (median bootstrap ARI {np.median(o['ari']):.2f}). The largest segment by AUM is **{biggest}** ({pct(p.loc[biggest,'aum_share'])}). Segments relate moderately to sector and client type (Cramér's V {o['validation'].set_index('attribute').cramers_v.iloc[0]:.2f} / {o['validation'].set_index('attribute').cramers_v.iloc[1]:.2f}) and not to country, so they partly recover sector without being a relabelling of it.
4. **Rare traders** ({int(rare.clients):,} clients) hold **{pct(rare.aum_share)}** of AUM, matching the 27% found last time. They trade about once every three years, so their history says little about how they would redeem, but **{int(c.top10_rare_traders)} of the top 10** and **{int(c.top100_rare_traders)} of the top 100** holders are rare traders.
5. **Stress barely changes the typical client, but it changes large redemptions.** The median client's stress lift is close to 1 in every segment ({lift.min():.2f}–{lift.max():.2f}): most clients redeem at their usual rate. Measured **by value** and compared with calm days in the same calendar month, {', '.join(f"{r.segment} {r.month_matched_value_lift:.2f}×" for r in vl.sort_values('month_matched_value_lift', ascending=False).head(3).itertuples())} redeemed more on stress days. These are driven by fewer, larger redemptions, so they are less certain, but rare traders alone hold {pct(rare.aum_share)} of AUM. The calendar (March and December year-ends, month-ends) remains the main driver of outflows.
6. **Segment flows are weakly correlated.** The mean pairwise correlation of daily flow rates is {cs[('daily','raw','all')]:.2f} raw and {cs[('daily','deseasonalised','all')]:.2f} after removing calendar effects. In stress it is {cs[('daily','raw','stress')]:.2f} raw and {cs[('daily','deseasonalised','stress')]:.2f} deseasonalised, so it does not rise materially at daily frequency. Weekly figures are higher ({cs[('weekly','raw','all')]:.2f} raw; {cs[('weekly','raw','stress')]:.2f} in stress weeks) because shared calendar timing accumulates within a week. Most of the co-movement is shared seasonality.
7. **Concentration is the dominant liquidity risk.** The top 10 clients hold {pct(c.top10_share)} and the top 100 hold {pct(c.top100_share)}, broadly stable since 2022. A full exit by the top 10 ({gbp(T.loc['Top 10 clients redeem in full','outflow_gbp'])}) is about {T.loc['Top 10 clients redeem in full','outflow_gbp']/w5:.0f}× the worst 5-day net outflow ever observed ({gbp(w5)} at today's AUM).
8. **Reverse stress.** A 10% liquid buffer, after £{o['base']['known_payables']/1e6:.1f}m of known payables, covers the full exit of only the **{rev['10% of AUM']} largest clients**; 20% covers {rev['20% of AUM']}. Historical and segment-tail scenarios (≤ {pct(max(T.loc[T.family.isin(['Historical replay','Segment tails']),'outflow_share']))} of AUM) stay well inside a 5% buffer.
'''
display(Markdown(txt))


1. **Data reconcile.** Client flows equal fund flows to the penny on trade and cash dates, and units × price rebuild fund AUM every day. No cancelled, duplicate, multi-currency or omnibus records were found. Internal moves cannot be identified without a flow-type code.
2. **Fund type.** Non-MMF (price-variable): annualised price volatility 10.3%, maximum drawdown -30.4%. Stress is defined from the price plus named episodes (781 stress days).
3. **Segments.** k = 7 behavioural segments plus rare traders. Silhouette is low (0.16): client behaviour is a continuum rather than sharply separated groups. Assignments are fairly stable (median bootstrap ARI 0.72). The largest segment by AUM is **Large active holders** (39.8%). Segments relate moderately to sector and client type (Cramér's V 0.35 / 0.36) and not to country, so they partly recover sector without being a relabelling of it.
4. **Rare traders** (2,834 clients) hold **27.3%** of AUM, matching the 27% found last time. They trade about once every three years, so their history says little about how they would redeem, but **6 of the top 10** and **24 of the top 100** holders are rare traders.
5. **Stress barely changes the typical client, but it changes large redemptions.** The median client's stress lift is close to 1 in every segment (0.93–1.02): most clients redeem at their usual rate. Measured **by value** and compared with calm days in the same calendar month, Low-activity holders 1.42×, Rare traders 1.36×, Month-end traders 1.20× redeemed more on stress days. These are driven by fewer, larger redemptions, so they are less certain, but rare traders alone hold 27.3% of AUM. The calendar (March and December year-ends, month-ends) remains the main driver of outflows.
6. **Segment flows are weakly correlated.** The mean pairwise correlation of daily flow rates is 0.11 raw and 0.05 after removing calendar effects. In stress it is 0.12 raw and 0.06 deseasonalised, so it does not rise materially at daily frequency. Weekly figures are higher (0.18 raw; 0.22 in stress weeks) because shared calendar timing accumulates within a week. Most of the co-movement is shared seasonality.
7. **Concentration is the dominant liquidity risk.** The top 10 clients hold 13.4% and the top 100 hold 40.7%, broadly stable since 2022. A full exit by the top 10 (£400.8m) is about 6× the worst 5-day net outflow ever observed (£62.7m at today's AUM).
8. **Reverse stress.** A 10% liquid buffer, after £11.9m of known payables, covers the full exit of only the **6 largest clients**; 20% covers 18. Historical and segment-tail scenarios (≤ 3.8% of AUM) stay well inside a 5% buffer.


### Comparison with the first dataset

The brief asks for a comparison with the first dataset. The project folder contains only one database (the charity fund above). Its rare-trader share (27%) matches the figure quoted from the earlier analysis, so this appears to be the **same** dataset. The table below is written so that a second database can be added: set `CONFIG_NEW` to that database's mapping and the same metrics are computed side by side. Until then, the second column shows the figures quoted from last time where they exist.

In [14]:
CONFIG_NEW = None   # e.g. {**S.CONFIG, 'name': 'New fund', 'files': {...}, 'flows': {...}}
def headline(r):
    p = r['profiles'].set_index('segment'); c = r['concentration'].iloc[-1]; cs = r['corr_summary'].set_index(['frequency','basis','regime']).mean_pairwise_r
    return {'Fund type': r['ctx']['fund_type'], 'Segments (k)': int(r['selection'].query('chosen').k.iloc[0]), 'Rare-trader AUM share': pct(p.loc['Rare traders','aum_share']),
            'Median stress lift (range across segments)': f"{p.stress_lift.min():.2f}–{p.stress_lift.max():.2f}", 'Mean daily segment correlation (raw / deseasonalised)': f"{cs[('daily','raw','all')]:.2f} / {cs[('daily','deseasonalised','all')]:.2f}",
            'Mean correlation in stress (daily, raw)': f"{cs[('daily','raw','stress')]:.2f}", 'Top-10 share': pct(c.top10_share), 'Top-100 share': pct(c.top100_share),
            'Largest clients covered by a 10% buffer': int(r['reverse'].set_index('buffer').clients_before_exhausted['10% of AUM'])}
cmp = pd.DataFrame({CONFIG['name']: headline(o)})
cmp['New database'] = pd.Series(headline(S.run(CONFIG_NEW, write=True, prefix='11new_'))) if CONFIG_NEW else pd.Series({'Rare-trader AUM share': f"{pct(CONFIG['previous']['rare_trader_aum_share'])} (quoted from last analysis)"})
display(cmp.reindex(list(headline(o))).fillna('—'))

,"Charity fund (synthetic, 2006–2026)",New database
Fund type,Non-MMF (price-variable),—
Segments (k),7,—
Rare-trader AUM share,27.3%,27.0% (quoted from last analysis)
Median stress lift (range across segments),0.93–1.02,—
Mean daily segment correlation (raw / deseasonalised),0.11 / 0.05,—
"Mean correlation in stress (daily, raw)",0.12,—
Top-10 share,13.4%,—
Top-100 share,40.7%,—
Largest clients covered by a 10% buffer,6,—
